In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura a renderização gráfica em linha (inline) do matplotlib para o notebook
%matplotlib inline

# Decodificar olhos abertos e olhos fechados a partir de EEG em repouso gravado

Carrega três participantes do Healthy Brain Network de ``ds005514``, identifica
os marcadores de instrução gravados de olhos abertos/fechados e avalia características
de potência na banda alfa em participantes retidos (*held-out*). Essas gravações têm
aproximadamente 95 MB cada; este exemplo específico da tarefa é maior do que o básico de SSVEP.
CPU é suficiente; retenha os downloads com ``EEGDASH_CACHE_DIR``. Instale
``eegprep[eeglabio]>=0.2.23,<0.3``. A limpeza com EEGPrep das gravações de densidade total
leva vários minutos e fica fora da pequena seleção de CI.
Veja o [conjunto de dados HBN](https://openneuro.org/datasets/ds005514).

A tarefa consiste em prever a condição ocular instruída de uma janela de EEG de dois segundos
a partir de um participante ausente do treinamento. Usamos a potência espectral integrada
em uma banda fixa de 8–13 Hz como um pequeno conjunto de características, mantendo a identidade
do participante ao longo do janelamento e da avaliação. Isso testa se essas características carregam
informações úteis sobre a condição nas gravações selecionadas.

Execute de cima para baixo com o EEGDash instalado; os tutoriais de pré-processamento e
divisão segura contra vazamento (*leakage-safe split*) apresentam os objetos Braindecode usados aqui.
As saídas são uma tabela de contagem sujeito-por-condição, um espectro descritivo
e uma pontuação retida por participante. Cortar (*crop*) essas gravações de origem
reduziria o processamento, mas não evitaria o download inicial.


## 1. Consultar uma coorte delimitada
IDs explícitos tornam a carga de trabalho de três gravações reprodutível. Mantenha os nomes
de canais EGI fornecidos pela fonte; substituí-los por nomes arbitrários do sistema 10–20
descaracterizaria a identidade dos eletrodos. Este pequeno subconjunto de cinco canais
é fixado antes da avaliação, e o gráfico usa E70, seu primeiro canal.



In [ ]:
# Importa utilitários do sistema operacional e manipulação de caminhos
import os
from pathlib import Path

# Importa bibliotecas para plotagem, computação numérica e manipulação de tabelas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa funções de pré-processamento e criação de janelas da Braindecode
from braindecode.preprocessing import (
    EEGPrep,
    Preprocessor,
    create_windows_from_events,
    preprocess,
)
# Importa modelo de classificação, métricas e divisão Leave-One-Group-Out do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset e utilitários de extração espectral do EEGDash
from eegdash import EEGDashDataset
from eegdash.features import spectral_bands_power, spectral_preprocessor
from eegdash.hbn.preprocessing import hbn_ec_ec_reannotation

# Define o diretório de cache a partir da variável de ambiente ou usa o padrão local '.eegdash_cache'
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Lista explícita de sujeitos a serem carregados para reprodutibilidade
subjects = ["NDARAE710YWG", "NDARAH239PGG", "NDARAL897CYV"]
# Inicializa e faz o download do dataset na tarefa de estado de repouso (RestingState)
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="ds005514",
    task="RestingState",
    subject=subjects,
    n_jobs=1,
)
# Garante que todos os 3 sujeitos foram carregados corretamente
assert len(dataset.datasets) == len(subjects)
# Subconjunto pré-definido de canais EGI e Cz para análise
channels = ["E70", "E62", "E92", "E96", "Cz"]
# Itera pelas gravações para verificar a presença dos canais e das anotações de instrução
for recording in dataset.datasets:
    raw = recording.raw
    # Valida que todos os canais selecionados existem na gravação atual
    assert set(channels) <= set(raw.ch_names)
    # Valida que as anotações esperadas de abrir e fechar os olhos estão presentes
    assert {"instructed_toCloseEyes", "instructed_toOpenEyes"} <= set(
        raw.annotations.description
    )
    # Exibe o identificador do sujeito e a contagem de cada anotação encontrada
    print(
        recording.description["subject"],
        pd.Series(raw.annotations.description).value_counts(),
    )

## 2. Limpar com EEGPrep e reter os tempos de instrução gravados
O EEGPrep realiza remoção de deslocamento/deriva (*offset/drift*), detecção de canais ruins,
reconstrução de picos (*bursts*) com ASR, interpolação de canais e referência média comum.
Execute-o na montagem completa de EEG antes de selecionar cinco preditores: verificações
espaciais de canais precisam da cobertura completa de eletrodos disponível. Reamostrar para
128 Hz reduz a computação deste estágio. Um filtro passa-baixa de 40 Hz segue o pipeline de limpeza.

O corte fixo de ASR de 20 é uma escolha para demonstração, não uma garantia de EEG livre de artefatos.
A limpeza é calibrada independentemente em cada gravação completa, incluindo uma gravação
retida (*held-out*) não rotulada. Este é um protocolo de calibração offline por gravação;
não é um limpador fixo aprendido apenas em sujeitos de treinamento. O classificador abaixo
ainda recebe rótulos apenas das pessoas do conjunto de treino.

A rejeição de janelas inteiras é desativada para que a limpeza mantenha a linha do tempo original.
As conversões entre MNE e EEGLAB do EEGPrep podem alterar o tempo de anotação e as datas de medição.
Salve as anotações de origem, verifique se apenas a taxa de amostragem foi alterada e, em seguida,
restaure seus tempos físicos. Não use essa restauração se habilitar qualquer operação que remova
segmentos de tempo.



In [ ]:
# Salva uma cópia de segurança das anotações originais de cada gravação
annotations = [recording.raw.annotations.copy() for recording in dataset.datasets]
# Armazena a data/hora original da medição de cada gravação
measurement_dates = [recording.raw.info["meas_date"] for recording in dataset.datasets]
# Calcula a duração física em segundos de cada gravação (número de amostras / taxa de amostragem)
durations = [
    recording.raw.n_times / recording.raw.info["sfreq"]
    for recording in dataset.datasets
]
# Garante que a primeira amostra de cada gravação comece no índice zero
assert all(recording.raw.first_samp == 0 for recording in dataset.datasets)
# Executa o pipeline de pré-processamento: reamostragem, limpeza com EEGPrep e filtro passa-baixa em 40 Hz
preprocess(
    dataset,
    [
        EEGPrep(
            resample_to=128,
            burst_removal_cutoff=20,
            bad_window_max_bad_channels=None,
            max_mem_mb=128,
        ),
        Preprocessor("filter", l_freq=None, h_freq=40),
    ],
    n_jobs=1,
)
# Restaura metadados temporais e aplica anotações específicas do protocolo HBN
for recording, annotation, measurement_date, duration in zip(
    dataset.datasets, annotations, measurement_dates, durations
):
    raw = recording.raw
    # Valida que o início continua em 0 e que a duração total após reamostragem a 128 Hz se manteve idêntica
    assert raw.first_samp == 0 and abs(raw.n_times / 128 - duration) <= 1 / 128
    # Restaura a data original da medição
    raw.set_meas_date(measurement_date)
    # Restaura as anotações originais
    raw.set_annotations(annotation)
    # Verifica numericamente que os tempos de início (onset) e descrições foram preservados com exatidão
    np.testing.assert_allclose(raw.annotations.onset, annotation.onset, atol=1e-12)
    np.testing.assert_array_equal(raw.annotations.description, annotation.description)
    # O utilitário HBN substitui anotações; retém explicitamente os intervalos ruins (BAD) originais.
    bad_spans = annotation[
        np.char.startswith(np.char.lower(annotation.description), "bad")
    ]
    # Aplica reanotação padrão HBN para olhos abertos / olhos fechados
    hbn_ec_ec_reannotation().apply(raw)
    # Adiciona os trechos ruins de volta às anotações da gravação
    raw.set_annotations(raw.annotations + bad_spans)
    # Seleciona apenas o subconjunto de 5 canais de interesse
    raw.pick(channels)

## 3. Janelar períodos estáveis após as instruções reais
O utilitário HBN define os inícios de 15 a 27 segundos após uma instrução de fechar os olhos
e de 5 a 17 segundos após uma instrução de abrir os olhos, a cada dois segundos. O deslocamento
final (*stop offset*) de 256 amostras estende esses marcadores de duração zero para janelas de
dois segundos a 128 Hz, cobrindo 15..29 e 5..19 segundos. Tamanho e passo (*stride*) iguais evitam
sobreposição. O alvo é a condição instruída, sem verificação independente de conformidade.
Intervalos BAD existentes são respeitados ao criar janelas.

O empilhamento de janelas do Braindecode cria matrizes de formato (janelas, 5 canais, 256 amostras)
em volts. A ordem das linhas de metadados fornece a condição observada e o grupo do participante
para cada janela. Inspecione as contagens reais retidas antes da modelagem.



In [ ]:
# Cria janelas de 2 segundos (256 amostras a 128 Hz) a partir dos eventos definidos
windows = create_windows_from_events(
    dataset,
    mapping={"eyes_open": 0, "eyes_closed": 1},
    trial_start_offset_samples=0,
    trial_stop_offset_samples=256,
    window_size_samples=256,
    window_stride_samples=256,
    on_last_window="drop",
    use_mne_epochs=True,  # O MNE rejeita épocas que se sobrepõem às anotações BAD preservadas.
    preload=True,
)
# Obtém a tabela de metadados associada a cada janela criada
metadata = windows.get_metadata()
# Empilha os dados das janelas em uma matriz numpy 3D (n_janelas, n_canais, n_amostras)
X = np.stack([window[0] for window in windows])
# Extrai os rótulos de condição (0 para aberto, 1 para fechado)
y = metadata["target"].to_numpy(dtype=int)
# Extrai o identificador do sujeito para cada janela como agrupador
groups = metadata["subject"].astype(str).to_numpy()
# Assegura dimensões corretas (5 canais, 256 amostras) e que não haja valores infinitos ou NaN
assert X.shape[1:] == (len(channels), 256) and np.isfinite(X).all()
# Garante que todos os 3 sujeitos estão representados nas janelas
assert set(groups) == set(subjects)
# Exibe tabela cruzada de contagem de janelas por sujeito e por condição
print(pd.crosstab(groups, y, rownames=["subject"], colnames=["condition"]))

## 4. Calcular a potência alfa com as funções espectrais do EEGDash
``spectral_preprocessor`` calcula uma PSD de Welch compartilhada entre as características e
o gráfico. Um segmento Hamming de 256 amostras a 128 Hz resulta em intervalos (*bins*) de 0.5 Hz.
A PSD retém o formato (janelas, canais, frequências) e unidades em V²/Hz.
A função de banda do EEGDash soma os bins no intervalo semiaberto [8, 13) Hz;
multiplicar pela largura do bin integra a densidade na potência em V².

Um valor de potência alfa por canal resulta em características de dimensão (janelas, 5). O logaritmo
comprime seu intervalo dinâmico, e um piso (*floor*) mínimo evita log(0). Essas operações por janela
não requerem ajuste (*fit*); o escalador é ajustado apenas dentro das divisões de treino.



In [ ]:
# Calcula a densidade espectral de potência (PSD) de Welch para cada janela no intervalo de 1 a 40 Hz
frequencies, psd = spectral_preprocessor(
    X,
    _metadata={"info": dataset.datasets[0].raw.info},
    fs=128,
    nperseg=256,
    noverlap=0,
    window="hamming",
    f_min=1,
    f_max=40,
)
# Calcula a soma da PSD na banda alfa definida entre 8 e 13 Hz
alpha_power = spectral_bands_power(frequencies, psd, bands={"alpha": (8, 13)})["alpha"]
# Multiplica pela resolução em frequência (bin width) para integrar e obter potência em V²
alpha_power *= frequencies[1] - frequencies[0]
# Aplica transformação log10 com piso de 1e-30 para estabilidade numérica
features = np.log10(np.maximum(alpha_power, 1e-30))
# Valida o formato final da matriz de características (n_janelas, 5 canais) e finitude dos valores
assert features.shape == (len(metadata), len(channels)) and np.isfinite(features).all()

## 5. Ajustar um escalador e classificador novos em cada partição LOSO
O sujeito retido (*held-out*) não fornece estatísticas de escalonamento. Cada sujeito é
retido uma vez, e todas as janelas desse participante permanecem juntas na mesma partição.
A regressão logística combina as cinco características de potência em log para uma decisão binária.
A acurácia balanceada (*balanced accuracy*) calcula a média da sensibilidade de olhos abertos e
olhos fechados, com uma referência de chance de 0.5 quando ambas as condições estão presentes.
As asserções verificam essas condições e a cobertura exata de um teste por sujeito; elas não
exigem que o modelo supere o acaso. Com dois participantes no treino em cada rodada,
há pouco suporte para seleção de hiperparâmetros, portanto os parâmetros são fixados com antecedência.



In [ ]:
# Inicializa lista para armazenar os resultados de cada partição
rows = []
# Vetor para rastrear quantas vezes cada amostra foi avaliada no conjunto de teste
counts = np.zeros(len(y), dtype=int)
# Executa validação cruzada deixando um grupo (sujeito) fora a cada rodada (LOSO)
for train, test in LeaveOneGroupOut().split(features, y, groups):
    # Garante que os sujeitos do treino e do teste são completamente disjuntos (sem vazamento)
    assert set(groups[train]).isdisjoint(groups[test])
    # Garante que ambas as classes (0 e 1) estão presentes tanto no treino quanto no teste
    assert set(y[train]) == set(y[test]) == {0, 1}
    # Cria pipeline com padronização e regressão logística
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    # Ajusta o modelo usando estritamente os dados de treino da partição atual
    model.fit(features[train], y[train])
    # Registra o sujeito de teste e a acurácia balanceada obtida
    rows.append(
        {
            "subject": groups[test][0],
            "balanced_accuracy": balanced_accuracy_score(
                y[test], model.predict(features[test])
            ),
        }
    )
    # Incrementa a contagem de testes para as amostras da partição de teste atual
    counts[test] += 1
# Assegura que todos os sujeitos foram avaliados e que cada amostra foi testada exatamente uma vez
assert len(rows) == len(subjects) and np.all(counts == 1)
# Monta e exibe a tabela de resultados finais
results = pd.DataFrame(rows)
print(results.to_string(index=False))

## 6. Comparar os espectros medidos e as pontuações retidas
Faça a média dentro de cada sujeito antes de calcular a média entre sujeitos. Diferenças
nesta pequena coorte não precisam necessariamente reproduzir um efeito de livro didático ou superar o acaso.
A PSD é calculada em V²/Hz; multiplicar por 1e12 converte a densidade exibida para µV²/Hz.
O espectro é descritivo e inclui todos os três participantes. As barras, por outro lado,
resumem previsões estritamente retidas (*held-out*).
Uma diferença no espectro médio não garante separação das janelas individuais, e janelas repetidas
não aumentam o número de participantes independentes além de três. Verifique se os maiores picos
estão dentro da banda selecionada de 8–13 Hz antes de chamá-los de atividade alfa. Um grande
pico fora da banda é motivo para inspecionar os canais brutos e a qualidade da gravação;
seu tamanho por si só não identifica uma fonte neural.



In [ ]:
# Cria figura com dois subgráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
# Itera pelas condições de olhos abertos (0) e olhos fechados (1)
for label, name in [(0, "eyes open"), (1, "eyes closed")]:
    # Calcula a média da PSD para o canal E70 (índice 0) em cada sujeito para a condição atual
    subject_psds = [
        psd[(groups == subject) & (y == label), 0].mean(axis=0) for subject in subjects
    ]
    # Plota a média geral entre sujeitos da densidade espectral em escala logarítmica (convertendo para µV²/Hz)
    axes[0].semilogy(frequencies, np.mean(subject_psds, axis=0) * 1e12, label=name)
# Configura eixos e legenda do gráfico espectral
axes[0].set(xlabel="Frequency (Hz)", ylabel="PSD at E70 (µV²/Hz)")
axes[0].legend()
# Plota o gráfico de barras com a acurácia balanceada obtida para cada sujeito retido
axes[1].bar(range(len(results)), results["balanced_accuracy"])
# Define rótulos do eixo X com os nomes dos sujeitos rotacionados a 45 graus
axes[1].set_xticks(range(len(results)), results["subject"], rotation=45, ha="right")
# Linha de referência tracejada correspondente ao nível de chance (50%)
axes[1].axhline(0.5, color="black", linestyle="--", label="Chance")
# Configura limites e títulos do gráfico de barras
axes[1].set(ylabel="Balanced accuracy", xlabel="Held-out subject", ylim=(0, 1))
axes[1].legend()
# Exibe a figura gerada
plt.show()

## 7. Estender a avaliação no nível do participante
Inspecione quais canais e artefatos transitórios (*bursts*) o EEGPrep modifica antes de
interpretar os espectros como evidência fisiológica. Em seguida, inspecione a temporização
dos intervalos de instrução retidos. Depois, adicione participantes sob o mesmo protocolo fixo
e examine a distribuição de pontuações entre eles. Se você alterar a seleção de canais ou a
faixa de frequência com base no desempenho, selecione-os usando participantes de validação
separados dentro de cada divisão de treino. Misturar aleatoriamente janelas desse participante
no treino apenas mediria o desempenho em uma pessoa já observada.

